# v4-3 TempoSAR + POLARIS (Colab, 1x T4)
Runtime -> Change runtime type -> **T4 GPU**. Then run the cells in order.
1. Mount Google Drive: every finished fold is saved there, so a disconnect loses nothing. Re-run all cells to resume.
2. Upload `sm_v4_temposar_bundle.zip` (from the project's `colab/` folder).
3. The run takes ~1.5-2 h: TempoSAR+POLARIS and the TempoSAR control run side by side on the one GPU.
4. The last cell zips the results. Download `sm_v4_temposar_results.zip` and put it in the project's `colab/` folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch the runtime to T4 GPU first"
from google.colab import drive
drive.mount("/content/drive")
import os
RES = "/content/drive/MyDrive/sm_v4_temposar/results_v2"
os.makedirs(RES + "/tables", exist_ok=True)
print("results ->", RES)

In [ ]:
import os, zipfile
from google.colab import files
B = "/content/sm"
if not os.path.exists(B + "/src_v2/exp_v4_temposar.py"):
    up = files.upload()                       # choose sm_v4_temposar_bundle.zip
    zipfile.ZipFile(next(iter(up))).extractall(B)
print(sorted(os.listdir(B)), sorted(os.listdir(B + "/src_v2")))

In [ ]:
import subprocess, sys, time, glob, os
env = dict(os.environ, SM_PROC=B + "/data/processed_v2", SM_RES=RES, PYTHONUNBUFFERED="1")
def launch(flag, name):
    cmd = " && ".join(f'"{sys.executable}" exp_v4_temposar.py --protocol "{p}" {flag}'
                      for p in ["leave-stations-out", "leave-one-region-out"])
    return subprocess.Popen(cmd, shell=True, cwd=B + "/src_v2", env=env,
                            stdout=open(f"{RES}/{name}.log", "a"), stderr=subprocess.STDOUT)
procs = {"polaris": launch("--polaris", "polaris"), "control": launch("", "control")}
t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(60)
    n_p = len(glob.glob(RES + "/tables/v4_temposar_polaris/*/f*.parquet"))
    n_c = len(glob.glob(RES + "/tables/v4_temposar/*/f*.parquet"))
    print(f"{(time.time() - t0) / 60:5.0f} min  +POLARIS {n_p}/19 folds   control {n_c}/19 folds", flush=True)
for k, p in procs.items():
    print(k, "exit", p.returncode)
    print(open(f"{RES}/{k}.log").read()[-1500:])

In [ ]:
import shutil
shutil.make_archive("/content/sm_v4_temposar_results", "zip", RES)
from google.colab import files
files.download("/content/sm_v4_temposar_results.zip")